In [14]:
## changing working directory

import os
os.getcwd()

os.chdir("..")
os.getcwd()

from IPython.display import display


In [6]:
import json
import pandas as pd
from collections import Counter

plan = pd.read_csv("notebooks/generation_plan.csv")

# load accepted samples
samples = []
with open("data/generated/raw/samples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        samples.append(json.loads(line))
df = pd.DataFrame(samples)

# count accepted per job
acc_counts = Counter(df["job_id"])

# merge with plan to see which jobs are short
plan["accepted"] = plan["job_id"].map(acc_counts).fillna(0).astype(int)
plan["missing"] = plan["n_samples"] - plan["accepted"]

incomplete = plan[plan["missing"] > 0].sort_values("missing", ascending=False)

print("Expected total:", plan["n_samples"].sum())
print("Accepted total:", plan["accepted"].sum())
print("Total missing:", plan["missing"].clip(lower=0).sum())
print("Incomplete jobs:", len(incomplete))

incomplete[["job_id","labels","anchor_regime","literal","specificity","consistent","n_samples","accepted","missing"]].head(20)


Expected total: 6912
Accepted total: 6892
Total missing: 20
Incomplete jobs: 7


,job_id,labels,anchor_regime,literal,specificity,consistent,n_samples,accepted,missing
999,9025212a-5665-43ca-8040-dfa2daf1c47f,temperature+pressure,strict,1,0,0,6,1,5
405,b0005033-4b4d-4fe2-862e-57360d978d17,temperature+vibration,strict,0,1,1,6,2,4
394,7ab28c65-c6f8-47fa-b75d-d9c0b3a17deb,temperature+vibration+pressure,strict,1,1,1,3,0,3
132,6058e392-6e87-4292-9859-0d8d0a29fbf1,nociception+temperature+pressure,strict,1,1,1,3,1,2
306,0aafc0a2-8c15-4412-bf56-24777ccedd6d,nociception+temperature+vibration,strict,1,1,1,3,1,2
325,07d23eaf-f7db-43ed-83f3-62c8c5ad82ec,nociception+vibration+pressure,strict,0,0,0,3,1,2
334,1e9cdf4c-c38e-4150-9d00-75a17ff65de0,temperature+vibration+pressure,strict,1,1,1,3,1,2


In [9]:
# sanity check for core label distributions 

display(df["literal"].value_counts(normalize=True))
display(df["specificity"].value_counts(normalize=True))
display(df["consistent"].value_counts(normalize=True))
display(df["anchor_regime"].value_counts())


literal
0    0.50058
1    0.49942
Name: proportion, dtype: float64

specificity
0    0.500435
1    0.499565
Name: proportion, dtype: float64

consistent
0    0.500435
1    0.499565
Name: proportion, dtype: float64

anchor_regime
drift         2304
paraphrase    2304
strict        2284
Name: count, dtype: int64

In [10]:
# modality combination frequency check
df["labels"].value_counts().sort_values(ascending=False)


labels
nociception                          864
pressure                             864
temperature                          864
vibration                            864
vibration+pressure                   432
nociception+vibration                432
nociception+temperature              432
nociception+pressure                 432
temperature+vibration                428
temperature+pressure                 427
nociception+temperature+pressure     214
nociception+vibration+pressure       214
nociception+temperature+vibration    214
temperature+vibration+pressure       211
Name: count, dtype: int64

In [11]:
# text length and structure check

df["text"].str.len().describe()

df.sample(10)[["text","labels","literal","specificity","consistent","anchor_regime"]]


,text,labels,literal,specificity,consistent,anchor_regime
2398,"I feel a light vibration pulse through me, yet...",vibration,1,0,0,strict
5943,A sharp throb in my lower back presses down an...,nociception+pressure,1,1,1,paraphrase
1799,A mild warmth in my forearm like ice melting a...,temperature,0,1,0,drift
2740,The chilling warmth spreads through me at once.,temperature,1,0,0,strict
4106,My muscles are aching while trembling spreads ...,nociception+vibration,1,0,1,strict
442,"It feels like a brief heating jab in my knee, ...",nociception+temperature+pressure,0,1,0,strict
61,A sharp sting spreads across my left forearm f...,nociception,1,1,1,drift
3325,A sharp sting in my hand overwhelms the icy ch...,nociception+temperature+pressure,1,1,0,paraphrase
6790,A steady vibration pulses through my fingertip...,vibration,1,1,1,drift
348,"A dull throb in my knee lasts for seconds, yet...",nociception,1,1,0,paraphrase
